**Complete worked solution.** Attempt the companion learning notebook first. All supplied cells here are executable.

# Comparing HDB resale prices across groups

**Question:** Are average resale prices different between two flat types in this teaching extract?

**Concepts:** Descriptive group comparison, Welch's independent-samples t-test, interpretation and limitations. **Data:** 2023 HDB teaching extract.

This is a comparison of recorded transactions, not a causal estimate of flat-type effects. The source Week 3 lab also examines regions and ANOVA; those extensions remain for a later revision.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Works in a repository checkout and remotely (including Colab).
name = "hdb_data_2023.csv.xz"
candidates = [
    Path("../datasets/singapore_hdb_ura") / name,
    Path("src/content/datasets/singapore_hdb_ura") / name,
    Path("datasets/singapore_hdb_ura") / name,
]
source = next((p for p in candidates if p.is_file()), "https://raw.githubusercontent.com/wcchin/geo-stat-vis/data/hdb-2013-2023-planning-boundaries/src/content/datasets/singapore_hdb_ura/" + name)
df = pd.read_csv(source, index_col=0)
assert {"resale_price", "resale_age"}.issubset(df.columns)
print(f"Loaded {len(df):,} records from {source}")
df[["resale_price", "resale_age"]].describe()

## 1. Inspect categories

Choose two flat types with adequate observations.

In [ ]:
print("Columns:", df.columns.tolist())
assert "flat_type" in df.columns, "Expected flat_type in the prepared HDB extract"
print(df["flat_type"].value_counts().head(10))
valid = df.dropna(subset=["flat_type", "resale_price"])
counts = valid["flat_type"].value_counts()
groups = counts.index[:2].tolist()
assert len(groups) == 2
a = valid.loc[valid["flat_type"] == groups[0], "resale_price"].astype(float)
b = valid.loc[valid["flat_type"] == groups[1], "resale_price"].astype(float)
print("Selected groups:", groups)
print(pd.DataFrame({"n":[len(a),len(b)], "mean":[a.mean(),b.mean()], "median":[a.median(),b.median()]},index=groups))

## 2. Exercise — visual comparison

**TODO:** Plot a labelled boxplot of the selected groups. Identify the direction of the difference and possible outliers.

In [ ]:
fig, ax = plt.subplots()
ax.boxplot([a, b], tick_labels=groups)
ax.set(xlabel="Flat type", ylabel="Resale price (SGD)", title="Price by flat type")
plt.show()

## 3. Exercise — inferential comparison

**TODO:** Conduct a two-sided Welch independent-samples t-test (`scipy.stats.ttest_ind`, `equal_var=False`). State the null hypothesis, p-value and a limitation of interpretation.

In [ ]:
from scipy.stats import ttest_ind
result = ttest_ind(a, b, equal_var=False, alternative="two-sided")
print("Welch t statistic:", result.statistic, "p-value:", result.pvalue)
print("Observed mean difference:", a.mean() - b.mean())

## Reflection

Would a very small p-value establish why the groups differ? What other variables might explain part of the observed price differences?

## Worked interpretation

The null is equality of population means under the sampling/model assumptions. A p-value does not measure the magnitude or cause of the observed difference. Differences in floor area, location, age and selection effects can matter.